# Vaidya — Phase 3: Accuracy Eval

Compares **base Mistral-7B-Instruct-v0.3** vs **fine-tuned (LoRA)** on 1000-row stratified sample of MedMCQA test set.

**Run this after Phase 2 training completes.**

Expected results:
- Base model: ~45–55% accuracy (random = 25%)
- Fine-tuned: ~60–70%+ after 1 epoch

## Step 1 — Install packages

In [ ]:
!pip install -q \
    transformers==4.47.0 \
    peft==0.14.0 \
    bitsandbytes \
    accelerate==0.34.2 \
    datasets==3.2.0 \
    mlflow==2.19.0 \
    boto3 \
    sentencepiece \
    "fsspec[http]==2024.9.0" \
    "packaging==24.2"

!pip install -q bitsandbytes --upgrade --force-reinstall

print('✅ Install done')
print('⚠️  RESTART THE KERNEL, then run from Step 2.')

## Step 2 — Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU'
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Step 3 — Set working directory

In [ ]:
import os, sys

# eval script lives in eval/, qlora_config.py in layer2_finetune/lightning/
repo_root = '/teamspace/studios/this_studio/vaidya'
eval_dir  = f'{repo_root}/eval'
os.chdir(eval_dir)
print(f'Working dir: {os.getcwd()}')

## Step 4 — AWS credentials

In [ ]:
import os, boto3

# ── Fill in your credentials ────────────────────────────────
os.environ['AWS_ACCESS_KEY_ID']     = 'YOUR_ACCESS_KEY_ID'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'YOUR_SECRET_ACCESS_KEY'
os.environ['AWS_DEFAULT_REGION']    = 'ap-south-1'
# ────────────────────────────────────────────────────────────

s3 = boto3.client('s3')
buckets = [b['Name'] for b in s3.list_buckets()['Buckets']]
assert 'vaidya-artifacts' in buckets, '❌ vaidya-artifacts not found'
print('✅ S3 connected')

## Step 5 — Run eval

Evaluates base + fine-tuned on 1000-row sample.  
~30–45 min per model on A100.  
Change `--run-name` to match your checkpoint (run1 = Kaggle, run2 = Lightning.ai A100).

In [ ]:
!python eval_accuracy.py --help

In [ ]:
# Full eval: base + fine-tuned
# Change --run-name to whichever checkpoint finished first (run1 or run2)
!python eval_accuracy.py --run-name run2 --n-samples 1000

## Step 6 — View results

In [ ]:
import json, pandas as pd

with open('eval_results.json') as f:
    r = json.load(f)

print('=== Accuracy ===')
if 'base' in r:
    print(f"Base model      : {r['base']['accuracy']*100:.1f}%  ({r['base']['correct']}/{r['base']['total']})")
if 'finetuned' in r:
    print(f"Fine-tuned      : {r['finetuned']['accuracy']*100:.1f}%  ({r['finetuned']['correct']}/{r['finetuned']['total']})")
if 'delta' in r:
    print(f"Delta           : {r['delta']*100:+.1f}%")

print()
print('=== Top subjects (fine-tuned) ===')
if 'finetuned' in r:
    subj = r['finetuned']['subject_accuracy']
    df = pd.DataFrame(subj.items(), columns=['Subject', 'Accuracy'])
    df = df.sort_values('Accuracy', ascending=False).head(10)
    df['Accuracy'] = df['Accuracy'].map(lambda x: f'{x*100:.1f}%')
    print(df.to_string(index=False))

## Gate check — before Phase 4

- ✅ `eval_results.json` has `base_accuracy`, `finetuned_accuracy`, `delta`
- ✅ Fine-tuned > base accuracy
- ✅ Results at `s3://vaidya-artifacts/eval/eval_results.json`
- ✅ Logged to MLflow (`vaidya-eval` experiment)

**These numbers go on your resume: "Improved MedMCQA accuracy from X% → Y% (+Z%)"**

**Next:** Phase 4 — DPO fine-tuning + adapter merge + HuggingFace Hub push